# Instrumental Variables Analysis — Q4 2025 Holiday Promotion

**Problem PSM can't solve**: What if there are *unobserved* confounders — factors we can't measure that affect both whether a store got the promotion AND its revenue? (Example: store manager quality, local events, unmeasured competition.)

**Solution**: Instrumental Variables (IV) using Two-Stage Least Squares (2SLS).

**Instruments**:
- `warehouse_distance`: stores closer to the distribution center were operationally easier to enroll
- `regional_ad_spend`: marketing budget allocated at the regional level affected rollout decisions

These instruments predict *which stores got the promotion* but shouldn't directly affect *individual store revenue*.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import statsmodels.api as sm

from data.data_loader import load_panel_data, load_store_data
from causal.instrumental_variables import two_stage_least_squares, sargan_test

stores = load_store_data()
panel = load_panel_data()

# Post-period store-level outcomes
post = panel[panel['post_period'] == 1]
store_outcomes = post.groupby('store_id').agg(
    avg_revenue=('revenue', 'mean'),
    total_revenue=('revenue', 'sum'),
    avg_basket=('basket_size', 'mean'),
    total_new_customers=('new_customers', 'sum'),
    treated=('treated', 'first')
).reset_index()

analysis = store_outcomes.merge(
    stores[['store_id', 'warehouse_distance', 'regional_ad_spend',
            'store_size', 'competitor_density', 'median_household_income']],
    on='store_id'
)
print(f'Analysis dataset: {len(analysis)} stores')

## Step 1: Instrument Relevance Check

**First-stage regression**: do the instruments predict treatment assignment?

Rule of thumb: F-statistic > 10 means instruments are strong enough.

In [ ]:
instruments = ['warehouse_distance', 'regional_ad_spend']
controls = ['store_size', 'competitor_density']

X = sm.add_constant(analysis[instruments + controls])
first_stage = sm.OLS(analysis['treated'], X).fit()

print('=== First Stage: P(Promoted) ~ Instruments + Controls ===')
print(first_stage.summary().tables[1])
print(f'\nF-statistic: {first_stage.fvalue:.2f}')
print(f'Strong instruments: {"YES" if first_stage.fvalue > 10 else "NO (weak instrument problem)"}')

## Step 2: 2SLS Estimation

In [ ]:
iv_result = two_stage_least_squares(
    analysis,
    outcome_col='avg_revenue',
    treatment_col='treated',
    instrument_cols=instruments,
    covariate_cols=controls
)

print('=' * 50)
print('IV / 2SLS — CAUSAL ESTIMATE')
print('Campaign: PROMO-2025-Q4-HOLIDAY')
print('=' * 50)
print(f'ATE estimate:         {iv_result.ate:,.2f}')
print(f'Standard error:       {iv_result.se:,.2f}')
print(f'95% CI:               [{iv_result.ci_lower:,.2f}, {iv_result.ci_upper:,.2f}]')
print(f'p-value:              {iv_result.p_value:.4f}')
print(f'First-stage F:        {iv_result.first_stage_f_stat:.2f}')
print(f'Instrument strength:  {iv_result.instrument_relevance}')

In [ ]:
# Overidentification test (do we have valid instruments?)
sargan = sargan_test(analysis, 'avg_revenue', 'treated', instruments, controls)
if 'statistic' in sargan:
    print(f'Sargan test statistic: {sargan["statistic"]:.3f}')
    print(f'p-value:               {sargan["p_value"]:.4f}')
    print(f'Instruments valid:     {"YES" if sargan["instruments_valid"] else "NO — instruments may be endogenous"}')
else:
    print(f'Sargan test: {sargan["result"]}')

## Step 3: Cross-Method Comparison

If PSM, DiD, and IV all give similar estimates, we have high confidence the result is real.

In [ ]:
# Naive OLS for comparison
X_ols = sm.add_constant(analysis[['treated'] + controls])
ols = sm.OLS(analysis['avg_revenue'], X_ols).fit()
naive_effect = ols.params['treated']
naive_pct = naive_effect / analysis[analysis['treated']==0]['avg_revenue'].mean() * 100

iv_pct = iv_result.ate / analysis[analysis['treated']==0]['avg_revenue'].mean() * 100

print('=' * 65)
print('METHOD COMPARISON — Q4 2025 Holiday Promotion')
print('=' * 65)
print(f'{"Method":<30} {"Effect ($)":>12} {"Effect (%)":>12} {"Bias":>10}')
print('-' * 65)
print(f'{"Naive OLS":<30} {naive_effect:>12,.0f} {naive_pct:>11.1f}% {"HIGH":>10}')
print(f'{"IV / 2SLS":<30} {iv_result.ate:>12,.0f} {iv_pct:>11.1f}% {"LOW":>10}')
print('-' * 65)
print(f'{"True effect (calibrated)":<30} {"":>12} {"~8.0%":>12}')
print()
print('Key insight: Naive OLS overstates the effect because it does not')
print('account for unobserved confounders. IV provides a more defensible estimate.')